Adapted from https://www.kaggle.com/code/beckpro/lightgbm-fold-voting-baseline-lb-0-408

In [ ]:
!mkdir -p /etc/OpenCL/vendors && echo "libnvidia-opencl.so.1" > /etc/OpenCL/vendors/nvidia.icd

In [ ]:
!pip install rdkit-pypi

In [ ]:
!pip install duckdb

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import OneHotEncoder

# Generate ECFPs
def generate_ecfp(molecule, radius=2, bits=1024):
    if molecule is None:
        return None
    return list(AllChem.GetMorganFingerprintAsBitVect(molecule, radius, nBits=bits))

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, ClassifierMixin
from tqdm.auto import tqdm
tqdm.pandas()
# wip
# import optuna 
train_path = '/kaggle/input/leash-BELKA/train.parquet'
test_path = '/kaggle/input/leash-BELKA/test.parquet'

targets = ['BRD4', 'sEH', 'HSA']
con = duckdb.connect()
data = {}

def get_data(target):
    df = con.query(f"""(SELECT molecule_smiles, binds
                            FROM parquet_scan('{train_path}')
                            WHERE binds = 0
                            and protein_name = '{target}'
                            ORDER BY random()
                            LIMIT 30000)
                            UNION ALL
                            (SELECT molecule_smiles, binds
                            FROM parquet_scan('{train_path}')
                            WHERE binds = 1
                            and protein_name = '{target}'
                            ORDER BY random()
                            LIMIT 30000)""").df()
    
    df['molecule'] = df['molecule_smiles'].progress_apply(Chem.MolFromSmiles)
    df['ecfp'] = df['molecule'].progress_apply(generate_ecfp)
    return df[['ecfp', 'binds']]

x = list(map(get_data, targets))

con.close()

In [ ]:
class VotingModel(BaseEstimator, ClassifierMixin):
    def __init__(self, estimators):
        super().__init__()
        self.estimators = estimators
        
    def fit(self, X, y=None):
        return self
    
    def predict(self, X):
        y_preds = [estimator.predict(X) for estimator in self.estimators]
        return np.mean(y_preds, axis=0)
    
    def predict_proba(self, X):
        y_preds = [estimator.predict_proba(X) for estimator in self.estimators]
        return np.mean(y_preds, axis=0)

In [ ]:
data = dict(zip(targets, x))

In [ ]:
assert(list(data.keys()) == targets)

In [ ]:
# Split the data into train and test sets
skf = StratifiedKFold(n_splits=5, shuffle=False)

params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": 'average_precision',
    "max_depth": 8,
    "learning_rate": 0.05,
    "n_estimators": 1000,
    "colsample_bytree": 0.8, 
    "colsample_bynode": 0.8,
    "verbose": -1,
    "random_state": 42,
    "device": "gpu",
}

def fit_models(df):
    X = pd.DataFrame(df['ecfp'].to_list())
    y = df['binds']
    
    fitted_models = []
    for idx_train, idx_valid in skf.split(X, y):
        X_train, y_train = X.iloc[idx_train], y.iloc[idx_train]
        X_valid, y_valid = X.iloc[idx_valid], y.iloc[idx_valid]
    
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[lgb.log_evaluation(100), lgb.early_stopping(100)]
        )

        fitted_models.append(model)

    return VotingModel(fitted_models)

models = {k: fit_models(v) for k, v in data.items()}

In [ ]:
import os


# Process the test.parquet file chunk by chunk
test_file = '/kaggle/input/leash-BELKA/test.csv'
output_file = 'submission.csv'  # Specify the path and filename for the output file

def predict(df_test):
    # Generate ECFPs for the molecule_smiles
    df_test['molecule'] = df_test['molecule_smiles'].progress_apply(Chem.MolFromSmiles)
    df_test['ecfp'] = df_test['molecule'].progress_apply(generate_ecfp)
    df_final = df_test.drop(columns = ['molecule', 'molecule_smiles'])
    
    df_final['binds'] = df_final.progress_apply(lambda x: models[x['protein_name']].predict_proba(np.array(x['ecfp']).reshape(1, -1))[:,1][0], axis = 1)
    return df_final[['id', 'binds']]

i = 0
# Read the test.csv file into a pandas DataFrame
for df in pd.read_csv(test_file, usecols = ['id', 'molecule_smiles', 'protein_name'], chunksize=100000):
    print(f'{i}')
    output_df = predict(df)
    # Save the output DataFrame to a CSV file
    output_df.to_csv(output_file, index=False, mode='a', header=not os.path.exists(output_file))
    i += 1